# 1. LIAR DATASETS 

In [ ]:
!pip install datasets pandas

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd

# datasets>=4.0 no longer runs loading scripts (liar.py). Two script-free options:
try:
    # 1) Hugging Face's auto-converted Parquet copy of the dataset (no script needed)
    ds = load_dataset("ucsbnlp/liar", revision="refs/convert/parquet")
except Exception as e:
    print(f"Parquet branch unavailable ({type(e).__name__}); falling back to original TSV files")
    # 2) Original LIAR release (Wang, 2017)
    !wget -q -nc https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
    !unzip -q -o liar_dataset.zip -d liar_dataset
    cols = ["id", "label", "statement", "subject", "speaker", "job_title", "state_info",
            "party_affiliation", "barely_true_counts", "false_counts", "half_true_counts",
            "mostly_true_counts", "pants_on_fire_counts", "context"]
    read = lambda f: pd.read_csv(f"liar_dataset/{f}", sep="\t", header=None, names=cols, quoting=3)
    ds = DatasetDict({
        "train": Dataset.from_pandas(read("train.tsv")),
        "validation": Dataset.from_pandas(read("valid.tsv")),
        "test": Dataset.from_pandas(read("test.tsv")),
    })

print(ds)
# DatasetDict with splits: train (~10.2k), validation (~1.3k), test (~1.3k)

print(ds["train"][0])

Parquet branch unavailable (DatasetNotFoundError); falling back to original TSV files
DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 10269
    })
    validation: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 1284
    })
    test: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 1283
    })
})
{'id': '2635.json', 'label': 'false', 's

In [ ]:
train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()

# The Parquet copy stores labels as integers (ClassLabel); the TSV fallback already has strings.
label_feature = ds["train"].features["label"]
if hasattr(label_feature, "names"):
    label_names = label_feature.names
    for df in (train_df, val_df, test_df):
        df["label_name"] = df["label"].map(lambda i: label_names[i])
else:
    label_names = sorted(train_df["label"].unique())
    for df in (train_df, val_df, test_df):
        df["label_name"] = df["label"]

print(label_names)
print(train_df[["statement", "speaker", "party_affiliation", "label_name"]].head())

['barely-true', 'false', 'half-true', 'mostly-true', 'pants-fire', 'true']
                                           statement         speaker  \
0  Says the Annies List political group supports ...    dwayne-bohac   
1  When did the decline of coal start? It started...  scott-surovell   
2  Hillary Clinton agrees with John McCain "by vo...    barack-obama   
3  Health care reform legislation is likely to ma...    blog-posting   
4  The economic turnaround started at the end of ...   charlie-crist   

  party_affiliation   label_name  
0        republican        false  
1          democrat    half-true  
2          democrat  mostly-true  
3              none        false  
4          democrat    half-true  


In [ ]:
def summary(df):
    print(f"There's {df.shape[0]} rows and {df.shape[1]} columns")
    print()
    print("-"*50)
    print("The columns types are :\n")
    print()
    df.info()
    print("-"*50)
    print()
    print("-"*50)
    print("The number of missing values is : ", df.isna().sum().sum())
    print()
    print("The percentage of nan in total  :\n", (df.isna().sum().sum() / len(df)) * 100)
    print()
    print("The number of missing values per column is :\n", df.isna().sum())
    print()
    print("The percentage of nan per column :\n", (df.isna().sum() / len(df)) * 100)
    print()
    print("-"*50)
    n_dup = df.shape[0] - df.drop_duplicates().shape[0]
    print(f"There's {n_dup} duplicates")
    if n_dup > 0:
        df.drop_duplicates(inplace=True)
        print("Duplicates deleted")
    print()
    print("-"*50)
    print("The stats of the numerical columns:\n", df.select_dtypes(include='number').describe())
    print()
    print("Number of categories for object/categorical data :\n", df.select_dtypes(include='object').nunique())


summary(train_df)

There's 10269 rows and 15 columns

--------------------------------------------------
The columns types are :


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10269 entries, 0 to 10268
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    10269 non-null  object
 1   label                 10269 non-null  object
 2   statement             10269 non-null  object
 3   subject               10269 non-null  object
 4   speaker               10269 non-null  object
 5   job_title             7367 non-null   object
 6   state_info            8058 non-null   object
 7   party_affiliation     10269 non-null  object
 8   barely_true_counts    10269 non-null  int64 
 9   false_counts          10269 non-null  int64 
 10  half_true_counts      10269 non-null  int64 
 11  mostly_true_counts    10269 non-null  int64 
 12  pants_on_fire_counts  10269 non-null  int64 
 13  context               10

,,,,,,,,,,proportion
id,label,statement,subject,speaker,job_title,state_info,party_affiliation,context,label_name,
100.json,true,"""He's sued gun manufacturers. He was supportive of Brady. He was supportive of things like assault weapon bans.""",guns,mike-huckabee,author,Arkansas,republican,an interview on MSNBC,true,0.000148
1000.json,pants-fire,"In the 1970s, the swine flu broke out . . . under another Democrat, President Jimmy Carter.",health-care,michele-bachmann,Congresswoman,Minnesota,republican,an interview with Pajamas Media.,pants-fire,0.000148
10000.json,half-true,"Dan Patrick said that if women get paid less than a man for the same job, that thats not a problem.","jobs,women",leticia-van-de-putte,Pharmacist,Texas,democrat,a debate in Austin,half-true,0.000148
10002.json,barely-true,President Barack Obama ordered our military to enlist illegal aliens.,"immigration,military",allen-west,"U.S. representative, Florida District 22",Florida,republican,a Facebook post,barely-true,0.000148
10003.json,barely-true,David Perdue opposes laws that provide equal pay protections for women.,"jobs,labor,women",georgians-together,PAC,Georgia,democrat,television attack ad,barely-true,0.000148
...,...,...,...,...,...,...,...,...,...,...
9990.json,barely-true,"We have a goal of 10 percent of [City of Providence] contracts going to women and minority businesses. In reality, less than 1 percent of contracts go to women and minority businesses.","city-government,debates,jobs,women",jorge-elorza,Law professor,Rhode Island,democrat,a radio debate,barely-true,0.000148
9991.json,barely-true,A 2011 measure passed into law with Wendy Davis support removes from the attorney general the ability to settle lawsuits just like the school funding case.,"corrections-and-updates,education,market-regulation",greg-abbott,governor,Texas,republican,a debate in the Rio Grande Valley.,barely-true,0.000148
9994.json,false,I recently had the opportunity to vote to give the president limited authority to begin strikes against terrorists in Iraq and Syria.,"foreign-policy,military,terrorism",bruce-braley,U.S. House of Representatives,Iowa,democrat,a debate,false,0.000148


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10269 entries, 0 to 10268
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    10269 non-null  object
 1   label                 10269 non-null  object
 2   statement             10269 non-null  object
 3   subject               10269 non-null  object
 4   speaker               10269 non-null  object
 5   job_title             7367 non-null   object
 6   state_info            8058 non-null   object
 7   party_affiliation     10269 non-null  object
 8   barely_true_counts    10269 non-null  int64 
 9   false_counts          10269 non-null  int64 
 10  half_true_counts      10269 non-null  int64 
 11  mostly_true_counts    10269 non-null  int64 
 12  pants_on_fire_counts  10269 non-null  int64 
 13  context               10169 non-null  object
 14  label_name            10269 non-null  object
dtypes: int64(5), object(10)
memory usage